In [3]:
import os
import json
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from category_encoders import TargetEncoder
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

# Constantes
TRAIN_TEST_SPLIT_SEED = 42
TEST_SIZE = 0.2  # 20% para test

def is_boolean_column(series):
    """Detección rápida de columnas booleanas usando numpy"""
    if pd.api.types.is_bool_dtype(series):
        return True
    # Convertir a numpy array para procesamiento más rápido
    unique_values = pd.unique(series.dropna().to_numpy())
    return set(unique_values).issubset({0, 1, 0.0, 1.0})

def identify_columns_for_processing(df):
    """Identifica eficientemente las columnas para procesar"""
    # Usar numpy para operaciones vectorizadas
    column_types = {}
    for col in df.columns:
        if col == 'nivel_triage':
            continue
        
        series = df[col]
        if is_boolean_column(series):
            column_types['boolean_cols'] = column_types.get('boolean_cols', []) + [col]
        elif np.issubdtype(series.dtype, np.number):
            column_types['numeric_cols'] = column_types.get('numeric_cols', []) + [col]
    
    return column_types

def batch_safe_transformation(data, cols):
    """Aplica transformaciones en lotes usando operaciones vectorizadas
    sin interacciones con el target"""
    transformations = {}
    
    # Convertir a numpy array para operaciones más rápidas
    data_array = data[cols].to_numpy()
    
    # Calcular máscaras una vez
    non_negative_mask = (data_array >= 0).all(axis=0)
    positive_mask = (data_array > 0).all(axis=0)
    
    # Aplicar transformaciones en lotes
    for i, col in enumerate(cols):
        if non_negative_mask[i]:
            transformations[f"{col}_log"] = np.log1p(data_array[:, i])
        if positive_mask[i]:
            transformations[f"{col}_sqrt"] = np.sqrt(data_array[:, i])
        
        # Transformaciones seguras para cualquier distribución
        transformations[f"{col}_squared"] = np.square(data_array[:, i])
        
        # Calcular z-score vectorizado
        mean = np.mean(data_array[:, i])
        std = np.std(data_array[:, i])
        if std > 0:  # Evitar división por cero
            transformations[f"{col}_zscore"] = (data_array[:, i] - mean) / std
    
    return pd.DataFrame(transformations, index=data.index)

def generate_feature_interactions(df_train, df_test, numeric_cols, target_col='nivel_triage'):
    """Genera interacciones de características seguras entre predictores
    sin leakage del target"""
    # Interacciones entre predictores (no con el target)
    interactions_train = {}
    interactions_test = {}
    
    # Limitar el número de interacciones para evitar explosión de dimensionalidad
    # Seleccionamos los top_k predictores más correlacionados con el target
    top_k = min(20, len(numeric_cols))
    if top_k > 1:
        # Usar correlación para encontrar las mejores columnas
        correlations = []
        for col in numeric_cols:
            if col in df_train.columns:
                corr = df_train[col].corr(df_train[target_col])
                if not np.isnan(corr):
                    correlations.append((col, abs(corr)))
        
        # Ordenar por correlación absoluta y tomar los top_k
        correlations.sort(key=lambda x: x[1], reverse=True)
        top_features = [x[0] for x in correlations[:top_k]]
        
        # Generar interacciones solo entre estas características top
        for i, col1 in enumerate(top_features):
            for col2 in top_features[i+1:]:
                # Multiplicación
                interaction_name = f"{col1}_times_{col2}"
                interactions_train[interaction_name] = df_train[col1] * df_train[col2]
                interactions_test[interaction_name] = df_test[col1] * df_test[col2]
                
                # Suma
                interaction_name = f"{col1}_plus_{col2}"
                interactions_train[interaction_name] = df_train[col1] + df_train[col2]
                interactions_test[interaction_name] = df_test[col1] + df_test[col2]
                
                # Diferencia
                interaction_name = f"{col1}_minus_{col2}"
                interactions_train[interaction_name] = df_train[col1] - df_train[col2]
                interactions_test[interaction_name] = df_test[col1] - df_test[col2]
    
    # Convertir a DataFrames
    interactions_train_df = pd.DataFrame(interactions_train, index=df_train.index)
    interactions_test_df = pd.DataFrame(interactions_test, index=df_test.index)
    
    return interactions_train_df, interactions_test_df

def apply_target_encoding(X_train, y_train, X_test, numeric_cols, cv=5):
    """Aplica target encoding con validación cruzada para evitar data leakage"""
    if len(numeric_cols) == 0:
        return pd.DataFrame(), pd.DataFrame()
        
    encoder = TargetEncoder(cols=numeric_cols, smoothing=10)
    
    # Ajustar el encoder en el conjunto de entrenamiento
    X_train_encoded = encoder.fit_transform(X_train[numeric_cols], y_train)
    
    # Transformar el conjunto de prueba con el encoder ajustado
    X_test_encoded = encoder.transform(X_test[numeric_cols])
    
    # Renombrar columnas para indicar que son encoded
    X_train_encoded.columns = [f"{col}_encoded" for col in X_train_encoded.columns]
    X_test_encoded.columns = [f"{col}_encoded" for col in X_test_encoded.columns]
    
    return X_train_encoded, X_test_encoded

def get_column_summary(stage, df, column_types=None):
    """Genera un resumen mejorado de las columnas con información de tipos"""
    summary = []
    summary.append(f"Resumen de columnas - {stage}")
    summary.append(f"Total columnas: {len(df.columns)}")
    summary.append(f"Muestras: {len(df)}")
    
    # Análisis de tipos
    type_counts = df.dtypes.value_counts()
    for dtype, count in type_counts.items():
        summary.append(f"  - {dtype}: {count}")
    
    # Información de tipos de columnas si está disponible
    if column_types:
        summary.append("\nDistribución de columnas:")
        summary.append(f"  - Booleanas: {len(column_types.get('boolean_cols', []))}")
        summary.append(f"  - Numéricas: {len(column_types.get('numeric_cols', []))}")
    
    # Análisis de completitud
    completeness = df.notna().mean().mean()
    summary.append(f"\nCompletitud promedio: {completeness:.2%}")
    
    return summary

def generate_pdf_report(summaries, output_path):
    """Genera un reporte PDF con los resúmenes de columnas."""
    pdf_path = os.path.join(output_path, "feature_engineering_report.pdf")
    doc = SimpleDocTemplate(pdf_path, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    # Título
    title_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontSize=24,
        spaceAfter=30
    )
    story.append(Paragraph("Reporte de Feature Engineering", title_style))
    story.append(Spacer(1, 20))

    # Fecha y hora
    date_style = ParagraphStyle(
        'DateStyle',
        parent=styles['Normal'],
        fontSize=12,
        spaceAfter=30
    )
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    story.append(Paragraph(f"Generado el: {current_time}", date_style))
    story.append(Spacer(1, 20))

    # Contenido principal
    for section_title, content in summaries.items():
        # Título de sección
        section_style = ParagraphStyle(
            'SectionTitle',
            parent=styles['Heading2'],
            fontSize=16,
            spaceAfter=12
        )
        story.append(Paragraph(section_title, section_style))
        story.append(Spacer(1, 10))

        # Contenido de la sección
        content_style = ParagraphStyle(
            'ContentStyle',
            parent=styles['Normal'],
            fontSize=12,
            leftIndent=20,
            spaceAfter=6
        )
        
        if isinstance(content, list):
            for line in content:
                story.append(Paragraph(str(line), content_style))
        else:
            story.append(Paragraph(str(content), content_style))
        
        story.append(Spacer(1, 20))

    # Generar PDF
    doc.build(story)
    return pdf_path

def main():
    # Cargar configuración
    base_path = os.path.dirname(os.getcwd())
    config_path = os.path.join(base_path, "config.json")
    
    with open(config_path, "r") as f:
        config = json.load(f)
    
    output_path = os.path.join(base_path, config["paths"]["intermediate"]["featured"])
    os.makedirs(output_path, exist_ok=True)
    
    # Inicializar diccionario de resúmenes
    summaries = {}
    
    # Cargar datos
    encoded_path = os.path.join(base_path, config["paths"]["intermediate"]["encoded"], "df_triage_encoded.parquet")
    df = pd.read_parquet(encoded_path)
    
    # Resumen inicial
    summaries["Estado Inicial"] = get_column_summary("Inicial", df)
    initial_columns = len(df.columns)
    
    # 1. Identificar columnas para procesar
    print("Identificando columnas...")
    column_types = identify_columns_for_processing(df)
    numeric_cols = column_types.get('numeric_cols', [])
    boolean_cols = column_types.get('boolean_cols', [])
    
    summaries["Clasificación de Columnas"] = get_column_summary("Post-clasificación", df, column_types)
    
    # NUEVO: División en conjuntos de entrenamiento y prueba desde el principio
    print("Dividiendo datos en conjuntos de entrenamiento y prueba...")
    X = df.drop(columns=['nivel_triage'])
    y = df['nivel_triage']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=TRAIN_TEST_SPLIT_SEED, stratify=y
    )
    
    # 2. Imputación por separado para train y test
    print("Realizando imputación...")
    
    # Ajustar imputador solo en datos de entrenamiento
    imputer = SimpleImputer(strategy='median')
    X_train_imputed_numeric = pd.DataFrame(
        imputer.fit_transform(X_train[numeric_cols]),
        columns=numeric_cols,
        index=X_train.index
    )
    
    # Transformar datos de prueba con el imputador ya ajustado
    X_test_imputed_numeric = pd.DataFrame(
        imputer.transform(X_test[numeric_cols]),
        columns=numeric_cols,
        index=X_test.index
    )
    
    # 3. Generación de características en lotes
    print("Generando características...")
    
    # Para conjunto de entrenamiento
    batch_size = 50  # Procesar 50 columnas a la vez
    train_new_features_dfs = []
    
    for i in range(0, len(numeric_cols), batch_size):
        batch_cols = numeric_cols[i:i + batch_size]
        batch_features = batch_safe_transformation(X_train_imputed_numeric, batch_cols)
        train_new_features_dfs.append(batch_features)
    
    train_new_features = pd.concat(train_new_features_dfs, axis=1)
    
    # Para conjunto de prueba
    test_new_features_dfs = []
    
    for i in range(0, len(numeric_cols), batch_size):
        batch_cols = numeric_cols[i:i + batch_size]
        batch_features = batch_safe_transformation(X_test_imputed_numeric, batch_cols)
        test_new_features_dfs.append(batch_features)
    
    test_new_features = pd.concat(test_new_features_dfs, axis=1)
    
    summaries["Generación de Features"] = [
        f"Features generadas: {len(train_new_features.columns)}",
        "Tipos de transformaciones:",
        "  - Logarítmicas (log1p)",
        "  - Raíz cuadrada",
        "  - Cuadráticas",
        "  - Z-score"
    ]
    
    # 4. Target encoding seguro con validación cruzada
    print("Aplicando target encoding...")
    encoded_train, encoded_test = apply_target_encoding(
        X_train_imputed_numeric, y_train, X_test_imputed_numeric, numeric_cols
    )
    
    # 5. Interacciones seguras entre predictores (no con el target)
    print("Generando interacciones...")
    interactions_train, interactions_test = generate_feature_interactions(
        X_train_imputed_numeric, X_test_imputed_numeric, numeric_cols, 'nivel_triage'
    )
    
    summaries["Interacciones"] = [
        f"Interacciones generadas: {len(interactions_train.columns)}",
        "Método: Interacciones entre predictores (sin usar el target)"
    ]
    
    # 6. Unir todos los conjuntos de características
    print("Uniendo conjuntos de características...")
    
    # Para entrenamiento
    train_features = pd.concat([
        X_train_imputed_numeric,
        train_new_features,
        encoded_train,
        interactions_train,
        X_train[boolean_cols]
    ], axis=1)
    
    # Para prueba
    test_features = pd.concat([
        X_test_imputed_numeric,
        test_new_features,
        encoded_test,
        interactions_test,
        X_test[boolean_cols]
    ], axis=1)
    
    # 7. Selección de características con SelectKBest
    print("Seleccionando características...")
    
    # Usar solo un porcentaje del conjunto de entrenamiento para hacer la selección más eficiente
    sample_size = min(10000, len(train_features))
    train_sample = train_features.sample(n=sample_size, random_state=42)
    y_sample = y_train.loc[train_sample.index]
    
    # Seleccionar características basadas en correlación con el target
    feature_selector = SelectKBest(f_classif, k='all')
    feature_selector.fit(train_sample, y_sample)
    
    # Obtener scores
    feature_scores = pd.DataFrame({
        'Feature': train_features.columns,
        'Score': feature_selector.scores_
    })
    
    # Ordenar por score y seleccionar las top 500 características
    feature_scores = feature_scores.sort_values('Score', ascending=False)
    top_features = feature_scores.head(500)['Feature'].tolist()
    
    # También eliminar características con varianza cercana a 0
    variance_selector = VarianceThreshold(threshold=0.01)
    variance_selector.fit(train_features[top_features])
    
    # Obtener las características finales después de ambas selecciones
    selected_features = [
        feature for i, feature in enumerate(top_features) 
        if i < len(variance_selector.get_support()) and variance_selector.get_support()[i]
    ]
    
    # 8. Dataset final
    df_train_feateng = pd.concat([
        pd.DataFrame(y_train, columns=['nivel_triage']),
        train_features[selected_features]
    ], axis=1)
    
    df_test_feateng = pd.concat([
        pd.DataFrame(y_test, columns=['nivel_triage']),
        test_features[selected_features]
    ], axis=1)
    
    # Unir para el dataset completo (solo para guardar)
    df_feateng = pd.concat([df_train_feateng, df_test_feateng])
    
    # Resumen final
    summaries["Resumen Final"] = get_column_summary("Final", df_feateng)
    
    summaries["Resumen Comparativo"] = [
        f"Columnas iniciales: {initial_columns}",
        f"Features generadas: {len(train_new_features.columns)}",
        f"Interacciones generadas: {len(interactions_train.columns)}",
        f"Características después de selección: {len(selected_features)}",
        f"Columnas finales: {len(df_feateng.columns)}"
    ]
    
    # Información de la división de datos
    summaries["División Train-Test"] = [
        f"Tamaño del conjunto de entrenamiento: {len(df_train_feateng)} muestras ({100*(1-TEST_SIZE):.0f}%)",
        f"Tamaño del conjunto de prueba: {len(df_test_feateng)} muestras ({100*TEST_SIZE:.0f}%)",
        f"Distribución de clases en entrenamiento: {y_train.value_counts().to_dict()}",
        f"Distribución de clases en prueba: {y_test.value_counts().to_dict()}"
    ]
    
    # Generar PDF
    print("Generando reporte PDF...")
    pdf_path = generate_pdf_report(summaries, output_path)
    
    # Guardar datasets
    print("Guardando datasets...")
    output_file = os.path.join(output_path, "df_feateng.parquet")
    pq.write_table(pa.Table.from_pandas(df_feateng), output_file)
    
    # También guardar los conjuntos de entrenamiento y prueba por separado
    train_output_file = os.path.join(output_path, "df_feateng_train.parquet")
    test_output_file = os.path.join(output_path, "df_feateng_test.parquet")
    
    pq.write_table(pa.Table.from_pandas(df_train_feateng), train_output_file)
    pq.write_table(pa.Table.from_pandas(df_test_feateng), test_output_file)
    
    print(f"\n✅ Feature engineering completado.")
    print(f"Dataset completo guardado en: {output_file}")
    print(f"Dataset de entrenamiento guardado en: {train_output_file}")
    print(f"Dataset de prueba guardado en: {test_output_file}")
    print(f"Reporte PDF generado en: {pdf_path}")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'numpy'